## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/Wildtype_mice/preprocessing")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

import matplotlib.pyplot as plt
import seaborn as sns

sc.settings.verbosity = 0             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(facecolor='white', color_map='cividis_r', dpi_save=300, dpi=100)

# load data

In [ ]:
adata = sc.read_h5ad(f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/wildtypes_raw_lognorm_scaled_batch_corrected_ptime_celltypes.h5ad')
adata

In [ ]:
htos = pd.read_csv(f"{DATA_ROOT}/external/hto_assignment.csv", index_col=0)
htos

# Compare annotations

In [ ]:
obs_mice = adata.obs.copy()

In [ ]:
obs_mice_sub = obs_mice[obs_mice.batch.isin(["3", "4"])]
obs_mice_sub

In [ ]:
# obs_mice["indexes"] = obs_mice.index.str.split("-").str[0]
htos["indexes"] = htos.index.str.split("-").str[0]
htos["batch"] = htos.index.str.split("-").str[1]
htos = htos[htos.batch.str.startswith("WT")]
htos

In [ ]:
htos.index = np.where(htos.batch.str.contains("20mo"), htos.indexes + "-3", htos.indexes + "-4")
htos

In [ ]:
obs_mice_htos = obs_mice_sub.join(htos, rsuffix="_fromHTOs")
obs_mice_htos

In [ ]:
obs_mice.batch.value_counts()

In [ ]:
htos.batch.value_counts()

In [ ]:
for celltype in obs_mice_htos.celltype.unique():
    print(f"--> {celltype}")
    print(obs_mice_htos[obs_mice_htos.celltype==celltype].hto_ass.value_counts())
    print("---------------------------------------------------")

# Assign the HTO to the full data

In [ ]:
adata.obs = adata.obs.join(obs_mice_htos[["hto_ass", "hto_zone", "hto_sum", "batch_fromHTOs"]])

In [ ]:
adata.obs = adata.obs.rename(columns={"batch_fromHTOs": "batch_name"})
adata.obs

In [ ]:
sc.pl.umap(adata, color=["celltype", "leiden", "batch_name", "hto_zone"])

In [ ]:
for batch in adata.obs.batch.unique():
    adata_batch = adata[adata.obs.batch == batch]
    sc.pl.umap(adata_batch, color=["celltype", "hto_zone"])

# Remove the qNSC2 ROB cell population from the lineage

In [ ]:
qnsc2 = adata[adata.obs.celltype == "qNSC2"].copy()
qnsc2

In [ ]:
sc.pl.umap(qnsc2, color="hto_zone")

In [ ]:
sc.tl.leiden(qnsc2, resolution=1)

In [ ]:
sc.pl.umap(qnsc2, color=["leiden", "hto_zone"])

In [ ]:
qnsc2_rob = qnsc2[qnsc2.obs.leiden=="5"].obs
qnsc2_rob

In [ ]:
adata.obs.celltype = adata.obs.celltype.cat.add_categories("ROB_qNSC2")
adata.obs.celltype

In [ ]:
adata.obs.loc[adata.obs.index.isin(qnsc2_rob.index), "celltype"] = "ROB_qNSC2"

In [ ]:
sc.pl.umap(adata, color=["celltype", "leiden", "batch_name", "hto_zone"], wspace=1, ncols=2)

In [ ]:
adata

# Write the file to disk

In [ ]:
adata.write_h5ad(f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/wildtypes_raw_lognorm_scaled_batch_corrected_ptime_celltypes_htos.h5ad')

# Check each celltype expression over pseudotime

In [ ]:
adata = sc.read_h5ad(f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/wildtypes_raw_lognorm_scaled_batch_corrected_ptime_celltypes_htos.h5ad')
adata

## Just the lineage

### Raw

In [ ]:
marker_genes_svz_lineage_sensible_dict = dict({
    "qNSC1": ["Aqp4", "Cxcl14"],
    "qNSC2": ["Cpe", "S100a6", "Sfrp1", "Notum"],
    "aNSC1": ["Btg2", ],
    "aNSC2": ["Ascl1", "Egfr", "Top2a"],
    "TAP": ["Mki67", "Ube2c"],
    "NB": ["Dcx", "Stmn2"],
})

marker_genes_svz_lineage_sensible_list = [i for f in marker_genes_svz_lineage_sensible_dict.values() for i in f]

In [ ]:
adata_raw = adata.raw.to_adata()
adata_lineage_raw = adata_raw[adata_raw.obs.celltype.isin(marker_genes_svz_lineage_sensible_dict.keys())]
adata_lineage_raw

In [ ]:
celltype_categories_reordered = [marker for marker in list(marker_genes_svz_lineage_sensible_dict.keys()) if marker in adata_lineage_raw.obs.celltype.cat.categories]
adata_lineage_raw.obs.celltype.cat.reorder_categories(celltype_categories_reordered, ordered=True, inplace=True)
adata_lineage_sorted = adata_lineage_raw[adata_lineage_raw.obs.sort_values(["celltype", "dpt_pseudotime"], ascending=True).index].copy()
adata_lineage_sorted.obs

In [ ]:
sc.settings.set_figure_params(dpi=120, facecolor='white')
sc.tl.dendrogram(adata_lineage_sorted, groupby="celltype", optimal_ordering=True)
sc.pl.tracksplot(adata_lineage_sorted, marker_genes_svz_lineage_sensible_dict, groupby='celltype', dendrogram=False)

### Lognorm

In [ ]:
sc.pp.normalize_per_cell(adata_lineage_raw)
sc.pp.log1p(adata_lineage_raw)

In [ ]:
celltype_categories_reordered = [marker for marker in list(marker_genes_svz_lineage_sensible_dict.keys()) if marker in adata_lineage_raw.obs.celltype.cat.categories]
adata_lineage_raw.obs.celltype.cat.reorder_categories(celltype_categories_reordered, ordered=True, inplace=True)
adata_lineage_sorted = adata_lineage_raw[adata_lineage_raw.obs.sort_values(["celltype", "dpt_pseudotime"], ascending=True).index].copy()

In [ ]:
sc.settings.set_figure_params(dpi=120, facecolor='white')
sc.tl.dendrogram(adata_lineage_sorted, groupby="celltype", optimal_ordering=True)
sc.pl.tracksplot(adata_lineage_sorted, marker_genes_svz_lineage_sensible_dict, groupby='celltype', dendrogram=False)

## With the ROB astrocytes appended

In [ ]:
marker_genes_lineage_plus_rob = ["Astrocytes", "ROB_qNSC2"] + list(marker_genes_svz_lineage_sensible_dict.keys())

In [ ]:
adata_raw = adata.raw.to_adata()
adata_lineage_raw = adata_raw[adata_raw.obs.celltype.isin(marker_genes_lineage_plus_rob)]
sc.pp.normalize_per_cell(adata_lineage_raw)
sc.pp.log1p(adata_lineage_raw)
adata_lineage_raw

In [ ]:
celltype_categories_reordered = [marker for marker in marker_genes_lineage_plus_rob if marker in adata_lineage_raw.obs.celltype.cat.categories]
adata_lineage_raw.obs.celltype.cat.reorder_categories(celltype_categories_reordered, ordered=True, inplace=True)
adata_lineage_sorted = adata_lineage_raw[adata_lineage_raw.obs.sort_values(["celltype", "dpt_pseudotime"], ascending=True).index].copy()

In [ ]:
sc.settings.set_figure_params(dpi=120, facecolor='white')
sc.tl.dendrogram(adata_lineage_sorted, groupby="celltype", optimal_ordering=True)
sc.pl.tracksplot(adata_lineage_sorted, marker_genes_svz_lineage_sensible_dict, groupby='celltype', dendrogram=False)

## Check expression of genes found by SPAPROS using these celltype labels
- The notebook with this analyses can be found at `$M26_DATA_ROOT/notebooks/Ifit1/5prime10X/select_probes_spapros_constrains_60probes.ipynb`

In [ ]:
marker_genes_svz_lineage_sensible_dict = dict({
    "qNSC1": ["Aqp4", "Cxcl14"],
    "qNSC2": ["Cpe", "S100a6", "Sfrp1", "Notum"],
    "aNSC1": ["Btg2", "Tnfrsf19"],
    "aNSC2": ["Ascl1", "Egfr", "Hes6","Cdk6", "Top2a"],
    "TAP": ["Mki67", "Ube2c", "Nudt4"],
    "NB": ["Dcx", "Stmn2"],
})

marker_genes_svz_lineage_sensible_list = [i for f in marker_genes_svz_lineage_sensible_dict.values() for i in f]

In [ ]:
adata_raw = adata.raw.to_adata()
adata_lineage_raw = adata_raw[adata_raw.obs.celltype.isin(marker_genes_svz_lineage_sensible_dict.keys())]
sc.pp.normalize_per_cell(adata_lineage_raw)
sc.pp.log1p(adata_lineage_raw)
adata_lineage_raw

In [ ]:
celltype_categories_reordered = [marker for marker in list(marker_genes_svz_lineage_sensible_dict.keys()) if marker in adata_lineage_raw.obs.celltype.cat.categories]
adata_lineage_raw.obs.celltype.cat.reorder_categories(celltype_categories_reordered, ordered=True, inplace=True)
adata_lineage_sorted = adata_lineage_raw[adata_lineage_raw.obs.sort_values(["celltype", "dpt_pseudotime"], ascending=True).index].copy()
adata_lineage_sorted.obs

In [ ]:
sc.settings.set_figure_params(dpi=120, facecolor='white')
sc.tl.dendrogram(adata_lineage_sorted, groupby="celltype", optimal_ordering=True)
sc.pl.tracksplot(adata_lineage_sorted, marker_genes_svz_lineage_sensible_dict, groupby='celltype', dendrogram=False)

# Extra: Check HVG max expression in pseudotime

In [ ]:
adata = sc.read_h5ad(f'{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/wildtypes_raw_lognorm_scaled_batch_corrected_ptime_celltypes_htos.h5ad')
adata

In [ ]:
adata = adata[~adata.obs.dpt_pseudotime.isna()]
adata

In [ ]:
adata.obs["celltype"] = adata.obs.celltype.cat.remove_unused_categories()

In [ ]:
adata.to_df()

In [ ]:
gene_expression_maxs = np.argmax(adata.X, axis=0)
gene_expression_maxs.shape

In [ ]:
ptimes = adata.obs.dpt_pseudotime
ptimes.shape

In [ ]:
gene_expression_maxs = [ptimes[i] for i in gene_expression_maxs]
len(gene_expression_maxs)

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(15, 9))
sns.kdeplot(gene_expression_maxs, bw_adjust=0.1, cut=0, ax=ax, c="dodgerblue")
sns.kdeplot(ptimes, bw_adjust=0.1, cut=0, ax=ax, c="firebrick")
sns.rugplot(data=adata.obs, x="dpt_pseudotime", hue="celltype", ax=ax, alpha=0.1)

ax.set_xlim((0, 1))
ax.set_xlabel("Pseudotime")

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(15, 4))
sns.kdeplot(gene_expression_maxs, bw_adjust=0.1, cut=0, ax=ax, c="dodgerblue")
sns.kdeplot(ptimes, bw_adjust=0.1, cut=0, ax=ax, c="firebrick")
sns.rugplot(data=adata.obs[adata.obs.celltype.isin(["Astrocytes"])], x="dpt_pseudotime", hue="celltype", ax=ax, alpha=0.1)

ax.set_xlim((0, 1))
ax.set_xlabel("Pseudotime")

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(15, 4))
sns.kdeplot(gene_expression_maxs, bw_adjust=0.1, cut=0, ax=ax, c="dodgerblue")
sns.kdeplot(ptimes, bw_adjust=0.1, cut=0, ax=ax, c="firebrick")
sns.rugplot(data=adata.obs[adata.obs.celltype.isin(["ROB_qNSC2"])], x="dpt_pseudotime", hue="celltype", ax=ax, alpha=0.1)

ax.set_xlim((0, 1))
ax.set_xlabel("Pseudotime")

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(15, 4))
sns.kdeplot(gene_expression_maxs, bw_adjust=0.1, cut=0, ax=ax, c="dodgerblue")
sns.kdeplot(ptimes, bw_adjust=0.1, cut=0, ax=ax, c="firebrick")
sns.rugplot(data=adata.obs[adata.obs.celltype.isin(["Neurons"])], x="dpt_pseudotime", hue="celltype", ax=ax, alpha=0.1)

ax.set_xlim((0, 1))
ax.set_xlabel("Pseudotime")

In [ ]:
adata_sub = adata[~adata.obs.celltype.isin(["Astrocytes", "ROB_qNSC2", "Neurons"])]

gene_expression_maxs = np.argmax(adata_sub.X, axis=0)
ptimes = adata_sub.obs.dpt_pseudotime
display(ptimes.shape)

gene_expression_maxs = [ptimes[i] for i in gene_expression_maxs]
len(gene_expression_maxs)

In [ ]:
obs = adata_sub.obs.copy()
obs.dpt_pseudotime.isna().any()

In [ ]:
obs["celltype"] = obs.celltype.cat.remove_unused_categories()
obs.celltype

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12, 5))
sns.rugplot(data=obs, x="dpt_pseudotime", hue="celltype", ax=ax, alpha=0.1, legend=True)
sns.kdeplot(gene_expression_maxs, bw_adjust=0.1, cut=0, ax=ax, c="dodgerblue", label="Gene expression maximum")
sns.kdeplot(ptimes, bw_adjust=0.1, cut=0, ax=ax, c="firebrick", label="Number of cells")

ax.set_xlim((0, 1))
ax.set_ylim((-0.3, None))
ax.set_xlabel("Pseudotime")
ax.grid(visible=False)

In [ ]:
h, l = ax.get_legend_handles_labels()
print(h, l, sep="\n")
plt.legend(h, l);
plt.grid(False)

In [ ]:
adata_sub_sub = adata_sub[:, adata_sub.var.highly_variable_intersection]

gene_expression_maxs = np.argmax(adata_sub_sub.X, axis=0)
ptimes = adata_sub_sub.obs.dpt_pseudotime
display(ptimes.shape)

gene_expression_maxs = [ptimes[i] for i in gene_expression_maxs]
len(gene_expression_maxs)

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12, 5))
sns.rugplot(data=adata_sub_sub.obs, x="dpt_pseudotime", hue="celltype", ax=ax, alpha=0.1, legend=True)
sns.kdeplot(gene_expression_maxs, bw_adjust=0.1, cut=0, ax=ax, c="dodgerblue", label="Gene expression maximum")
sns.kdeplot(ptimes, bw_adjust=0.1, cut=0, ax=ax, c="firebrick", label="Number of cells")

ax.set_xlim((0, 1))
ax.set_ylim((-0.3, None))
ax.set_xlabel("Pseudotime")
ax.grid(visible=False)

In [ ]:
adata_sub_sub = adata_sub[:, adata_sub.var.highly_variable_intersection]

gene_expression_mins = np.argmin(adata_sub_sub.X, axis=0)
ptimes = adata_sub_sub.obs.dpt_pseudotime
display(ptimes.shape)

gene_expression_mins = [ptimes[i] for i in gene_expression_mins]
len(gene_expression_maxs)

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12, 5))
sns.rugplot(data=adata_sub_sub.obs, x="dpt_pseudotime", hue="celltype", ax=ax, alpha=0.1, legend=True)
sns.kdeplot(gene_expression_mins, bw_adjust=0.1, cut=0, ax=ax, c="dodgerblue", label="Gene expression maximum")
sns.kdeplot(ptimes, bw_adjust=0.1, cut=0, ax=ax, c="firebrick", label="Number of cells")

ax.set_xlim((0, 1))
ax.set_ylim((-0.3, None))
ax.set_xlabel("Pseudotime")
ax.grid(visible=False)

In [ ]:
adata_sub_sub = adata_sub[:, adata_sub.var.highly_variable_intersection]

gene_expression_maxs = np.argmax(adata_sub_sub.X, axis=0)
gene_expression_mins = np.argmin(adata_sub_sub.X, axis=0)
ptimes = adata_sub_sub.obs.dpt_pseudotime
display(ptimes.shape)

gene_expression_maxs = [ptimes[i] for i in gene_expression_maxs]
gene_expression_mins = [ptimes[i] for i in gene_expression_mins]

display(len(gene_expression_maxs))
display(len(gene_expression_mins))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 5))
sns.rugplot(
    data=adata_sub_sub.obs,
    x="dpt_pseudotime",
    hue="celltype",
    ax=ax,
    alpha=0.1,
    legend=True,
)
sns.kdeplot(
    gene_expression_maxs,
    bw_adjust=0.1,
    cut=0,
    ax=ax,
    c="dodgerblue",
    linestyle="--",
    label="Gene expression maximum",
)
sns.kdeplot(
    gene_expression_mins,
    bw_adjust=0.1,
    cut=0,
    ax=ax,
    c="darkviolet",
    linestyle="--",
    label="Gene expression minimum",
)
sns.kdeplot(ptimes, bw_adjust=0.1, cut=0, ax=ax, c="orangered", label="Number of cells")

ax.set_xlim((0, 1))
ax.set_ylim((-0.3, None))
ax.set_xlabel("Pseudotime")
ax.grid(visible=False)

In [ ]:
h, l = ax.get_legend_handles_labels()
print(h, l, sep="\n")
plt.legend(h, l);
plt.grid(False)

# Batch/Mouse cell coverage

In [ ]:
sc.pl.umap(adata_sub, color=["celltype"])

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(10, 5))
axs = axs.flatten()
mice = adata_sub.obs.batch.unique()

for ax, mouse in zip(axs, mice):
    adata_mouse = adata_sub[adata_sub.obs.batch == mouse].copy()
    sc.pl.umap(adata_mouse, color=["dpt_pseudotime"], s=10, ax=ax, show=False)
    ax.set_title(f"Mouse {mouse}")

fig.tight_layout()

# Store the metadata as csv

In [ ]:
adata.obs

In [ ]:
adata.obs.to_csv(f"{DATA_ROOT}/3utr/wildtypes_analysis/cellranger/wildtypes_raw_lognorm_scaled_batch_corrected_ptime_celltypes_htos_OBS.csv")